# Day 12 结构特征方案设计

本 notebook 只做结构特征方案设计，不训练模型、不调参、不做阈值分析、不生成 processed 特征矩阵。目标是把 Day 10 / Day 11 发现的结构信号整理成 Day 13 可执行的实验方案。

## 1. 读取配置和 Day 10 / Day 11 输出

字段是匿名字段，不能虚构具体物理含义。前缀只作为匿名结构分组线索。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.features.structural_feature_design import (
    build_structural_feature_design_table,
    load_structural_feature_config,
    select_missing_indicator_candidates,
)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
structural_config = load_structural_feature_config(PROJECT_ROOT / "config" / "structural_features.yaml")

In [2]:
feature_distribution = pd.read_csv(cfg.metrics_dir / "day10_feature_distribution_summary.csv")
missing_zero_summary = pd.read_csv(cfg.metrics_dir / "day10_missing_zero_summary.csv")
pos_neg_diff = pd.read_csv(cfg.metrics_dir / "day10_pos_neg_distribution_diff.csv")
drift_summary = pd.read_csv(cfg.metrics_dir / "day10_train_valid_test_drift_summary.csv")
prefix_group_summary = pd.read_csv(cfg.metrics_dir / "day11_prefix_group_summary.csv")
prefix_signal_ranking = pd.read_csv(cfg.metrics_dir / "day11_prefix_group_signal_ranking.csv")
prefix_members = pd.read_csv(cfg.tables_dir / "day11_prefix_feature_members.csv")

feature_distribution.shape, prefix_group_summary.shape, prefix_members.shape

((510, 22), (107, 23), (170, 3))

## 2. 回顾 Day 10 字段级发现

In [3]:
train_missing_zero = missing_zero_summary.query("dataset == 'train_inner'")

display(train_missing_zero.sort_values("missing_rate", ascending=False).head(8))
display(train_missing_zero.sort_values("zero_rate", ascending=False).head(10))
display(pos_neg_diff.sort_values("missing_rate_diff_abs", ascending=False).head(10))

,dataset,feature_name,missing_rate,zero_rate,near_zero_rate,non_missing_count
78,train_inner,br_000,0.821625,0.030625,0.030625,8562
77,train_inner,bq_000,0.813021,0.029437,0.029437,8975
76,train_inner,bp_000,0.796750,0.027312,0.027312,9756
75,train_inner,bo_000,0.773875,0.025375,0.025375,10854
1,train_inner,ab_000,0.771479,0.183542,0.183542,10969
112,train_inner,cr_000,0.771479,0.227292,0.227292,10969
74,train_inner,bn_000,0.735437,0.024021,0.024021,12699
73,train_inner,bm_000,0.661375,0.020688,0.020688,16254


,dataset,feature_name,missing_rate,zero_rate,near_zero_rate,non_missing_count
27,train_inner,as_000,0.010688,0.988938,0.988938,47487
29,train_inner,au_000,0.010688,0.988250,0.988250,47487
6,train_inner,ag_000,0.010958,0.985792,0.985792,47474
41,train_inner,ay_009,0.010937,0.980000,0.980000,47475
32,train_inner,ay_000,0.010937,0.979563,0.979563,47475
7,train_inner,ag_001,0.010958,0.976667,0.976667,47474
33,train_inner,ay_001,0.010937,0.970104,0.970104,47475
34,train_inner,ay_002,0.010937,0.969562,0.969562,47475
35,train_inner,ay_003,0.010937,0.968063,0.968063,47475
51,train_inner,az_009,0.010937,0.958396,0.958396,47475


,feature_name,neg_missing_rate,pos_missing_rate,missing_rate_diff_abs,neg_zero_rate,pos_zero_rate,zero_rate_diff_abs,neg_median,pos_median,median_diff_abs,neg_p95,pos_p95,p95_diff_abs,neg_p99,pos_p99,p99_diff_abs,standardized_median_diff
78,br_000,0.833983,0.09250,0.741483,0.031059,0.00500,0.026059,320170.0,303110.0,17060.0,1310700.0,652640.0,658060.0,1310700.0,1310700.0,0.0,0.044705
77,bq_000,0.825297,0.08875,0.736547,0.029852,0.00500,0.024852,302420.0,301420.0,1000.0,1310700.0,668652.0,642048.0,1310700.0,1310700.0,0.0,0.002693
76,bp_000,0.808835,0.08375,0.725085,0.027712,0.00375,0.023962,281920.0,307440.0,25520.0,1310700.0,655956.0,654744.0,1310700.0,1310700.0,0.0,0.070562
75,bo_000,0.785657,0.07875,0.706907,0.025784,0.00125,0.024534,264680.0,310460.0,45780.0,1310700.0,686496.0,624204.0,1310700.0,1310700.0,0.0,0.132057
74,bn_000,0.746610,0.07625,0.670360,0.024428,0.00000,0.024428,246790.0,313660.0,66870.0,1310700.0,673490.0,637210.0,1310700.0,939100.4,371599.6,0.205453
73,bm_000,0.671335,0.07375,0.597585,0.021038,0.00000,0.021038,235500.0,320400.0,84900.0,1310700.0,672540.0,638160.0,1310700.0,859040.0,451660.0,0.289978
137,di_000,0.059089,0.51750,0.458411,0.768136,0.32875,0.439386,0.0,0.0,0.0,44520.0,8544581.0,8500061.0,612006.4,15094010.0,14482003.6,0.000000
136,dh_000,0.059110,0.51750,0.458390,0.824470,0.35250,0.471970,0.0,0.0,0.0,320.0,41685.0,41365.0,9153.7,237896.5,228742.8,0.000000
139,dk_000,0.059110,0.51750,0.458390,0.935763,0.47750,0.458263,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.000000
138,dj_000,0.059110,0.51750,0.458390,0.937797,0.48000,0.457797,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


Day 10 的核心启发：缺失规模、零值模式和长尾结构都可能是匿名工业数据中的结构信号，但这些信号必须在 Day 13 的 validation 流程中验证。

## 3. 回顾 Day 11 前缀组发现

In [4]:
display(prefix_group_summary["feature_count"].value_counts().sort_index().rename("prefix_group_count"))
display(prefix_group_summary.query("feature_count > 1")[
    ["prefix", "feature_count", "avg_missing_rate", "avg_zero_rate", "avg_abs_pos_neg_zero_diff", "avg_abs_skew"]
].sort_values("avg_zero_rate", ascending=False))

feature_count
1     100
10      7
Name: prefix_group_count, dtype: int64

,prefix,feature_count,avg_missing_rate,avg_zero_rate,avg_abs_pos_neg_zero_diff,avg_abs_skew
1,ay,10,0.010937,0.680919,0.086121,53.456561
0,ag,10,0.010958,0.511665,0.226608,47.638014
4,cn,10,0.011229,0.327723,0.204883,30.607857
2,az,10,0.010937,0.288060,0.086070,59.004227
5,cs,10,0.010917,0.139450,0.055856,60.849442
3,ba,10,0.011250,0.135825,0.095347,18.760063
6,ee,10,0.010937,0.118619,0.048231,15.599665


Day 11 的核心启发：107 个前缀组中，100 个是单字段前缀，真正适合做前缀聚合的多字段组只有 `ag`、`ay`、`az`、`ba`、`cn`、`cs`、`ee`。

## 4. 结构特征设计原则

- 本轮只设计，不评估效果。
- 所有筛选规则只能基于 `train_inner` fit。
- `valid` 用于 Day 13 选择结构特征方案和阈值。
- `official test` 只用于最终评估，不能反向筛特征。
- 不做 SVM / PCA / GridSearch / SHAP。
- 不解释匿名字段的真实物理含义。

## 5. 样本级缺失统计设计

设计：`sample_missing_count`、`sample_missing_rate`、`sample_non_missing_count`。这些特征描述单个样本整体数据完整性，不依赖字段物理含义。

## 6. 样本级零值统计设计

设计：`sample_zero_count`、`sample_zero_rate`、`sample_non_zero_count`。Day 10 显示大量字段零值率很高，因此样本级零值结构值得在 Day 13 验证。

## 7. 前缀组缺失聚合设计

候选前缀：高优先级 `ag`、`ay`、`cn`；中优先级 `az`、`cs`；低优先级 `ba`、`ee`。

设计：`prefix_<prefix>_missing_count`、`prefix_<prefix>_missing_rate`。prefix membership 只能来自 train schema 或 Day 11 成员表。

## 8. 前缀组零值聚合设计

候选前缀：`ag`、`ay`、`cn`、`az`、`cs`。

设计：`prefix_<prefix>_zero_count`、`prefix_<prefix>_zero_rate`。其中 `ay`、`ag`、`cn` 是 Day 13 第一轮重点观察方向。

## 9. 筛选后的 missing indicators 设计

候选字段只基于 train_inner 计算：`missing_rate >= 0.01` 且 `abs(pos_missing_rate - neg_missing_rate) >= 0.10`，最多保留 30 个。

In [5]:
missing_indicator_candidates = select_missing_indicator_candidates(
    missing_zero_summary,
    pos_neg_diff,
    structural_config,
)
missing_indicator_candidates.head(15)

,feature_name,missing_rate,missing_rate_diff_abs
0,br_000,0.821625,0.741483
1,bq_000,0.813021,0.736547
2,bp_000,0.796750,0.725085
3,bo_000,0.773875,0.706907
4,bn_000,0.735437,0.670360
5,bm_000,0.661375,0.597585
6,di_000,0.066729,0.458411
7,dh_000,0.066750,0.458390
8,dj_000,0.066750,0.458390
9,dk_000,0.066750,0.458390


## 10. 可选异常/长尾结构特征设计

候选：`sample_outlier_count_p99`、`sample_outlier_rate_p99`、`prefix_az_outlier_count`、`prefix_cs_outlier_count`。

这些特征需要从 train_inner 分位数计算阈值。Day 13 第一轮暂不启用，作为第二轮备选。

## 11. 输出结构特征设计表

In [6]:
design_table = build_structural_feature_design_table(
    missing_zero_summary=missing_zero_summary,
    pos_neg_diff=pos_neg_diff,
    prefix_group_summary=prefix_group_summary,
    structural_config=structural_config,
)

display(design_table.groupby(["feature_family", "priority", "use_in_day13_first_round"]).size().rename("row_count"))
display(design_table.head(20))

feature_family               priority  use_in_day13_first_round
optional_outlier_summary     low       False                        4
prefix_missing_summary       high      True                         6
                             low       True                         4
                             medium    True                         4
prefix_zero_summary          high      True                         6
                             medium    True                         4
sample_missing_summary       high      True                         3
sample_zero_summary          high      True                         3
selected_missing_indicators  high      True                        30
Name: row_count, dtype: int64

,feature_family,feature_name,source_columns_or_prefix,calculation_method,fit_data,transform_data,selection_rule,priority,expected_signal,risk_or_limitation,use_in_day13_first_round
0,sample_missing_summary,sample_missing_count,all numeric features,count missing values per row,train_inner only for numeric feature list,train_inner/valid/official_test with same feat...,no supervised selection,high,sample-level data completeness,may reflect general data quality rather than A...,True
1,sample_missing_summary,sample_missing_rate,all numeric features,missing count / numeric feature count,train_inner only for numeric feature list,train_inner/valid/official_test with same feat...,no supervised selection,high,sample-level data completeness,may reflect general data quality rather than A...,True
2,sample_missing_summary,sample_non_missing_count,all numeric features,numeric feature count - missing count,train_inner only for numeric feature list,train_inner/valid/official_test with same feat...,no supervised selection,high,sample-level data completeness,may reflect general data quality rather than A...,True
3,sample_zero_summary,sample_zero_count,all numeric features,count zero values per row,train_inner only for numeric feature list,train_inner/valid/official_test with same feat...,no supervised selection,high,sample-level zero pattern,zero may represent normal inactive state or an...,True
4,sample_zero_summary,sample_zero_rate,all numeric features,zero count / numeric feature count,train_inner only for numeric feature list,train_inner/valid/official_test with same feat...,no supervised selection,high,sample-level zero pattern,zero may represent normal inactive state or an...,True
5,sample_zero_summary,sample_non_zero_count,all numeric features,numeric feature count - zero count,train_inner only for numeric feature list,train_inner/valid/official_test with same feat...,no supervised selection,high,sample-level zero pattern,zero may represent normal inactive state or an...,True
6,prefix_missing_summary,prefix_ag_missing_count,"ag_000,ag_001,ag_002,ag_003,ag_004,ag_005,ag_0...",count missing values within ag group,prefix membership from train_inner schema,train_inner/valid/official_test with same pref...,prefix selected from Day11 multi-feature prefi...,high,ag group missing structure,"anonymous prefix, no physical meaning",True
7,prefix_missing_summary,prefix_ag_missing_rate,"ag_000,ag_001,ag_002,ag_003,ag_004,ag_005,ag_0...",missing count / ag group feature count,prefix membership from train_inner schema,train_inner/valid/official_test with same pref...,prefix selected from Day11 multi-feature prefi...,high,ag group missing structure,"anonymous prefix, no physical meaning",True
8,prefix_missing_summary,prefix_ay_missing_count,"ay_000,ay_001,ay_002,ay_003,ay_004,ay_005,ay_0...",count missing values within ay group,prefix membership from train_inner schema,train_inner/valid/official_test with same pref...,prefix selected from Day11 multi-feature prefi...,high,ay group missing structure,"anonymous prefix, no physical meaning",True
9,prefix_missing_summary,prefix_ay_missing_rate,"ay_000,ay_001,ay_002,ay_003,ay_004,ay_005,ay_0...",missing count / ay group feature count,prefix membership from train_inner schema,train_inner/valid/official_test with same pref...,prefix selected from Day11 multi-feature prefi...,high,ay group missing structure,"anonymous prefix, no physical meaning",True


## 12. Day 13 实验矩阵建议

第一轮建议只做：

- `median_all`
- `median_all_sample_missing`
- `median_all_sample_zero`
- `median_all_prefix_missing`
- `median_all_prefix_zero`
- `median_all_selected_missing_indicators`
- `median_all_structural_all`

暂不做：PCA、SVM、大规模 GridSearch、SHAP、official test 反向筛选，也暂不把 outlier 特征放进第一轮。

## 13. 中文小结

Day 12 将 Day 10 / Day 11 的结构信号整理成 6 类特征家族：样本级缺失统计、样本级零值统计、前缀组缺失聚合、前缀组零值聚合、筛选后的 missing indicators、可选异常/长尾统计。

Day 13 第一轮应优先验证 A/B/C/D/E 和结构特征组合，不应直接进入复杂调参。所有规则必须在 train_inner 上 fit，valid 用于选择方案和阈值，official test 只用于最终评估。